# 🚀 FASE 3: Pemodelan Baseline (Random Forest & XGBoost)
## Prediksi PM2.5 di Cekungan Bandung
**Kerja Praktik — PRSDI BRIN 2026**

---

Notebook ini berfokus pada pelatihan model *Machine Learning* klasik (Ensemble Trees) yaitu **Random Forest** dan **XGBoost**. Model ini berfungsi sebagai tolok ukur (baseline) yang kuat sebelum kita beranjak ke *Deep Learning* (LSTM).

Model XGBoost ini nantinya juga akan kita gunakan untuk analisis **SHAP (Explainable AI)** di Fase 5.


## 1. Import Library
Pastikan Anda sudah menginstall `xgboost`. Jika belum, buka terminal dan jalankan `pip install xgboost scikit-learn`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import joblib

# Evaluasi
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Model
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

plt.rcParams.update({'figure.figsize': (14, 6), 'font.size': 11})
print("[OK] Library loaded.")


## 2. Load Data Processed & Scaler
Memuat data yang sudah bersih dan terskala dari hasil notebook `02_Preprocessing`.


In [ ]:
DATA_DIR = r'D:\Kuliah Praktik\KP BRIN\data\processed'

# Load Dataset
train_scaled = pd.read_csv(os.path.join(DATA_DIR, 'train_scaled.csv'), index_col=0, parse_dates=True)
test_scaled = pd.read_csv(os.path.join(DATA_DIR, 'test_scaled.csv'), index_col=0, parse_dates=True)

# Load Scaler Target (Y) untuk keperluan inversi nanti
scaler_y = joblib.load(os.path.join(DATA_DIR, 'scaler_y.pkl'))

# Pisahkan Fitur (X) dan Target (y)
target_col = 'pm25'
X_train = train_scaled.drop(columns=[target_col])
y_train = train_scaled[target_col]

X_test = test_scaled.drop(columns=[target_col])
y_test = test_scaled[target_col]

print(f"X_train shape : {X_train.shape}")
print(f"X_test shape  : {X_test.shape}")
display(X_train.head(3))


## 3. Fungsi Evaluasi (Inversi)
Karena model memprediksi angka 0-1, kita harus mengubahnya kembali ke $\mu g/m^3$ menggunakan fungsi `inverse_transform` sebelum menghitung akurasi.


In [ ]:
def evaluate_model(model_name, y_true_scaled, y_pred_scaled):
    # 1. Inversi skala 0-1 kembali ke ug/m3
    # Ubah format array numpy agar sesuai dengan input scaler (reshape)
    y_true_inv = scaler_y.inverse_transform(y_true_scaled.values.reshape(-1, 1)).flatten()
    y_pred_inv = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    
    # 2. Hitung Metrik
    rmse = np.sqrt(mean_squared_error(y_true_inv, y_pred_inv))
    mae = mean_absolute_error(y_true_inv, y_pred_inv)
    r2 = r2_score(y_true_inv, y_pred_inv)
    
    # 3. Print Laporan
    print(f"=== Kinerja Model: {model_name} ===")
    print(f"  RMSE : {rmse:.2f} ug/m3")
    print(f"  MAE  : {mae:.2f} ug/m3")
    print(f"  R^2  : {r2:.4f}")
    
    return y_true_inv, y_pred_inv


## 4. Melatih Random Forest Regressor


In [ ]:
print("Melatih Random Forest (bisa memakan waktu 10-30 detik)...")
# Inisialisasi Model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# Training Model
rf_model.fit(X_train, y_train)

# Prediksi di Data Test
y_pred_rf = rf_model.predict(X_test)

# Evaluasi
y_true_inv, y_pred_rf_inv = evaluate_model("Random Forest", y_test, y_pred_rf)


## 5. Melatih XGBoost Regressor


In [ ]:
print("Melatih XGBoost (sangat cepat)...")
# Inisialisasi Model
xgb_model = xgb.XGBRegressor(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=6, 
    random_state=42, 
    n_jobs=-1
)

# Training Model
xgb_model.fit(X_train, y_train)

# Prediksi di Data Test
y_pred_xgb = xgb_model.predict(X_test)

# Evaluasi
_, y_pred_xgb_inv = evaluate_model("XGBoost", y_test, y_pred_xgb)


## 6. Visualisasi Perbandingan (Time-Series)
Kita akan melihat seberapa akurat model mengikuti alur naik-turun PM2.5 aktual pada 2 minggu pertama di data Test (September 2023).


In [ ]:
# Ambil 14 Hari pertama (14 * 24 jam = 336 baris) dari data test
subset_len = 336

time_axis = y_test.index[:subset_len]
actual = y_true_inv[:subset_len]
rf_pred = y_pred_rf_inv[:subset_len]
xgb_pred = y_pred_xgb_inv[:subset_len]

plt.figure(figsize=(16, 6))
plt.plot(time_axis, actual, label='Aktual (PM2.5)', color='black', linewidth=2)
plt.plot(time_axis, rf_pred, label='Prediksi Random Forest', color='blue', alpha=0.7, linestyle='--')
plt.plot(time_axis, xgb_pred, label='Prediksi XGBoost', color='red', alpha=0.7, linestyle='-.')

plt.title("Perbandingan Prediksi Baseline vs Aktual (2 Minggu Pertama Set Test)")
plt.ylabel("PM2.5 Concentration (ug/m3)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 7. Menyimpan Model
Menyimpan model ke dalam folder `models` untuk digunakan ulang di Fase 5 (SHAP).


In [ ]:
MODEL_DIR = r'D:\Kuliah Praktik\KP BRIN\models'
os.makedirs(MODEL_DIR, exist_ok=True)

# Simpan Random Forest menggunakan Joblib
joblib.dump(rf_model, os.path.join(MODEL_DIR, 'rf_baseline.pkl'))

# Simpan XGBoost (direkomendasikan menggunakan format bawaan XGBoost JSON)
xgb_model.save_model(os.path.join(MODEL_DIR, 'xgb_baseline.json'))

print(f"[OK] Model berhasil disimpan di folder: {MODEL_DIR}")
